In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Face Detection Testing\n",
    "Test and visualize face detection on our sample videos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Setup\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "import cv2\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from pathlib import Path\n",
    "from preprocessing.video_utils import VideoProcessor\n",
    "from preprocessing.face_detection import FaceDetector\n",
    "\n",
    "%matplotlib inline\n",
    "\n",
    "processor = VideoProcessor()\n",
    "detector = FaceDetector(method='mediapipe', confidence_threshold=0.5)\n",
    "\n",
    "print('✅ Face detection system ready!')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load sample videos\n",
    "real_dir = Path('../data/raw/sample_dataset/real')\n",
    "fake_dir = Path('../data/raw/sample_dataset/fake')\n",
    "\n",
    "real_videos = processor.find_videos(real_dir)\n",
    "fake_videos = processor.find_videos(fake_dir)\n",
    "\n",
    "print(f'Real videos: {len(real_videos)}')\n",
    "print(f'Fake videos: {len(fake_videos)}')\n",
    "\n",
    "# Test face detection on sample frames\n",
    "if real_videos:\n",
    "    sample_frames = processor.extract_frames(real_videos[0], max_frames=3)\n",
    "    print(f'\\nExtracted {len(sample_frames)} frames for testing')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize face detection results\n",
    "if 'sample_frames' in locals() and sample_frames:\n",
    "    fig, axes = plt.subplots(2, 3, figsize=(15, 10))\n",
    "    fig.suptitle('Face Detection Results')\n",
    "    \n",
    "    for i, frame in enumerate(sample_frames[:3]):\n",
    "        # Detect faces\n",
    "        faces = detector.detect_faces(frame)\n",
    "        \n",
    "        # Original frame\n",
    "        axes[0, i].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))\n",
    "        axes[0, i].set_title(f'Original Frame {i+1}')\n",
    "        axes[0, i].axis('off')\n",
    "        \n",
    "        # Frame with face detection\n",
    "        frame_with_faces = detector.visualize_detections(frame, faces)\n",
    "        axes[1, i].imshow(cv2.cvtColor(frame_with_faces, cv2.COLOR_BGR2RGB))\n",
    "        axes[1, i].set_title(f'Detected: {len(faces)} face(s)')\n",
    "        axes[1, i].axis('off')\n",
    "        \n",
    "        print(f'Frame {i+1}: Found {len(faces)} faces')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\nelse:\n",
    "    print('No frames available for testing')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Extract and display face crops\n",
    "if 'sample_frames' in locals() and sample_frames:\n",
    "    face_crops = []\n",
    "    \n",
    "    for i, frame in enumerate(sample_frames):\n",
    "        faces = detector.detect_faces(frame)\n",
    "        \n",
    "        for j, face_bbox in enumerate(faces):\n",
    "            face_crop = detector.extract_face(frame, face_bbox, target_size=(224, 224))\n",
    "            if face_crop is not None:\n",
    "                face_crops.append({\n",
    "                    'frame': i+1,\n",
    "                    'face': j+1,\n",
    "                    'crop': face_crop,\n",
    "                    'bbox': face_bbox\n",
    "                })\n",
    "    \n",
    "    print(f'Extracted {len(face_crops)} face crops')\n",
    "    \n",
    "    # Display face crops\n",
    "    if face_crops:\n",
    "        cols = min(4, len(face_crops))\n",
    "        rows = (len(face_crops) + cols - 1) // cols\n",
    "        \n",
    "        fig, axes = plt.subplots(rows, cols, figsize=(12, 3*rows))\n",
    "        if rows == 1:\n",
    "            axes = [axes] if cols == 1 else axes\n",
    "        else:\n",
    "            axes = axes.flatten()\n",
    "        \n",
    "        for i, face_data in enumerate(face_crops):\n",
    "            if i < len(axes):\n",
    "                axes[i].imshow(cv2.cvtColor(face_data['crop'], cv2.COLOR_BGR2RGB))\n",
    "                axes[i].set_title(f\"Frame {face_data['frame']}, Face {face_data['face']}\")\n",
    "                axes[i].axis('off')\n",
    "        \n",
    "        # Hide empty subplots\n",
    "        for i in range(len(face_crops), len(axes)):\n",
    "            axes[i].axis('off')\n",
    "        \n",
    "        plt.tight_layout()\n",
    "        plt.show()\n",
    "else:\n",
    "    print('No face crops extracted')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test face detection on both real and fake videos\n",
    "detection_stats = {'real': [], 'fake': []}\n",
    "\n",
    "# Test real videos\n",
    "for video_path in real_videos[:2]:  # Test first 2 videos\n",
    "    frames = processor.extract_frames(video_path, max_frames=5)\n",
    "    total_faces = 0\n",
    "    \n",
    "    for frame in frames:\n",
    "        faces = detector.detect_faces(frame)\n",
    "        total_faces += len(faces)\n",
    "    \n",
    "    detection_stats['real'].append({\n",
    "        'video': video_path.name,\n",
    "        'frames_tested': len(frames),\n",
    "        'total_faces': total_faces,\n",
    "        'avg_faces_per_frame': total_faces / len(frames) if frames else 0\n",
    "    })\n",
    "\n",
    "# Test fake videos\n",
    "for video_path in fake_videos[:2]:  # Test first 2 videos\n",
    "    frames = processor.extract_frames(video_path, max_frames=5)\n",
    "    total_faces = 0\n",
    "    \n",
    "    for frame in frames:\n",
    "        faces = detector.detect_faces(frame)\n",
    "        total_faces += len(faces)\n",
    "    \n",
    "    detection_stats['fake'].append({\n",
    "        'video': video_path.name,\n",
    "        'frames_tested': len(frames),\n",
    "        'total_faces': total_faces,\n",
    "        'avg_faces_per_frame': total_faces / len(frames) if frames else 0\n",
    "    })\n",
    "\n",
    "# Print results\n",
    "print('Face Detection Statistics:')\n",
    "print('\\nReal Videos:')\n",
    "for stat in detection_stats['real']:\n",
    "    print(f\"  {stat['video']}: {stat['total_faces']} faces in {stat['frames_tested']} frames (avg: {stat['avg_faces_per_frame']:.2f})\")\n",
    "\n",
    "print('\\nFake Videos:')\n",
    "for stat in detection_stats['fake']:\n",
    "    print(f\"  {stat['video']}: {stat['total_faces']} faces in {stat['frames_tested']} frames (avg: {stat['avg_faces_per_frame']:.2f})\")\n",
    "\n",
    "print('\\n✅ Face detection testing complete!')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",\n",
   "language": "python",\n",
   "name": "python3"\n",
  }\n },\n "nbformat": 4,\n "nbformat_minor": 4\n}\nEOF